#04 — Construction of the Label Graph
## Relational multilabel modeling with clinical co-occurrence

**Purpose of this notebook:**
1. Build the 3 variants of the label graph: G_emp (raw empirical), G_pmi (normalized nPMI), G_clin (clinically cured)
2. Sparsify and normalize adjacency matrices
3. Visualize the graphs (PT and EN) — figures for the article
4. Export matrices ready for consumption by relational models (NB04)
5. Consolidate everything into JSON

**Outputs:**
- `project/graphs/` — `.npy` and `.csv` adjacency matrices per variant
- `project/results/04_graph_stats.json` — statistics and metadata
- `project/figs/` — individual PT and EN views

In [ ]:
import json
import hashlib
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx

# ── Paths ────────────────────────────────── ──────────────────────────────────
ROOT        = Path(r"/workspace")
DATA_DIR    = ROOT / "Dev" / "Data"
IMGS_DIR    = DATA_DIR / "Imgs"
CSV_PATH    = DATA_DIR / "Imgs-anotadas" / "dataset_labels.csv"

OUT_DIR     = ROOT / 'project'
SPLITS_DIR  = OUT_DIR / "splits"
GRAPHS_DIR  = OUT_DIR / "graphs"
RESULTS_DIR = OUT_DIR / "results"
FIGS_DIR    = OUT_DIR / "figs"

GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

# ── Label config ────────────────────────────── ───────────────────────────────
CORE_COLS = ["ENANTEMA", "PÓLIPO", "ÚLCERA", "EROSÃO", "MICRONODULARIDADE"]
K = len(CORE_COLS)

LABEL_EN = {
    "ENANTEMA": "ENANTHEMA",
    "PÓLIPO":   "POLYP",
    "ÚLCERA":   "ULCER",
    "EROSÃO":   "EROSION",
    "MICRONODULARIDADE": "MICRONODULARITY"
}

print(f"Core labels: {CORE_COLS}")
print(f"K = {K} nodes in the graph")

In [ ]:
# ── Load training splits - one graph per seed
# Graph construction uses Fold 0 training set as a proxy to avoid test leakage.

# Doing this on fold 0 is sufficient to capture global co-occurrence distributions.

LABEL_COLS = ["NORMAL","ALTERADO","SALIVA","LUZ","ENANTEMA",
              "PÓLIPO","ÚLCERA","EROSÃO","MICRONODULARIDADE",
              "ECTASIA VASCULAR","NEOPLASIA"]

FOLDS = 5

train_splits = {}
for i in range(FOLDS):
    df = pd.read_csv(SPLITS_DIR / f"fold_{i}_train.csv")
    for col in CORE_COLS:
        df[col] = df[col].fillna(0).astype(int)
    train_splits[i] = df
    n = len(df)
    print(f"Fold {i}: {n} training images")

# Reference split for single-graph displays (seed 42)
df_train = train_splits[0]
N_train  = len(df_train)
print(f"\nReference split (Fold 0): {N_train} images")
print("Prevalence:")
for col in CORE_COLS:
    n = df_train[col].sum()
    print(f"  {col:22s}: {n:4d} ({n/N_train*100:.1f}%)")

## 1. G_emp — Empirical graph (normalized raw co-occurrence)

In [ ]:
# ── 1.1 Raw co-occurrence matrix ────────────────────── ───────────────────────
data_np = df_train[CORE_COLS].values.astype(float)  # (N, K)

coocc_raw = np.zeros((K, K), dtype=float)
for i in range(K):
    for j in range(K):
        coocc_raw[i, j] = float(((data_np[:, i] == 1) & (data_np[:, j] == 1)).sum())

print("Raw co-occurrence matrix (training set only):")
df_raw_coocc = pd.DataFrame(coocc_raw, index=CORE_COLS, columns=CORE_COLS)
print(df_raw_coocc.to_string())

# Marginal counts (diagonal = per-label count)
marginals = np.array([df_train[c].sum() for c in CORE_COLS], dtype=float)
print(f"\nMarginals: {dict(zip(CORE_COLS, marginals.astype(int)))}")

In [ ]:
# ── 1.2 Build G_emp ───────────────────────────── ─────────────────────────────
# Normalize off-diagonal by row marginal: A_emp[i,j] = P(j|i) = n_ij / n_i
# Diagonal = 1 (self-loop, required by GCN renormalization trick)
# This is the ML-GCN formulation (Chen et al., CVPR 2019).

A_emp = np.zeros((K, K), dtype=float)
for i in range(K):
    for j in range(K):
        if i == j:
            A_emp[i, j] = 1.0
        else:
            A_emp[i, j] = coocc_raw[i, j] / marginals[i] if marginals[i] > 0 else 0.0

print("G_emp adjacency (P(j|i), diagonal=1):")
print(pd.DataFrame(A_emp, index=CORE_COLS, columns=CORE_COLS).round(3).to_string())

## 2. G_pmi — Normalized nPMI graph

In [ ]:
# ── 2.1 Compute nPMI for all pairs ───────────────────── ──────────────────────
# nPMI(i,j) = PMI(i,j) / -log P(i,j) ∈ [-1, 1]
# PMI(i,j) = log2( P(i,j) / (P(i)*P(j)) )
# Laplace smoothing alpha=1 to handle zero counts.
# Symmetric: nPMI(i,j) = nPMI(j,i)

ALPHA = 1.0

npmi_mat = np.zeros((K, K), dtype=float)
pmi_raw  = np.zeros((K, K), dtype=float)

for i in range(K):
    for j in range(K):
        if i == j:
            npmi_mat[i, j] = 1.0
            continue
        n11 = coocc_raw[i, j]
        n_i = marginals[i]
        n_j = marginals[j]
        N   = float(N_train)

        # Smoothed probabilities
        p_i  = (n_i  + ALPHA) / (N + 2 * ALPHA)
        p_j  = (n_j  + ALPHA) / (N + 2 * ALPHA)
        p_ij = (n11  + ALPHA) / (N + 4 * ALPHA)

        pmi  = np.log2(p_ij / (p_i * p_j))
        npmi = pmi / (-np.log2(p_ij)) if p_ij < 1.0 else 0.0

        pmi_raw[i, j]  = round(pmi,  4)
        npmi_mat[i, j] = round(npmi, 4)

print("nPMI matrix (diagonal=1, off-diagonal ∈ [-1,1]):")
print(pd.DataFrame(npmi_mat, index=CORE_COLS, columns=CORE_COLS).round(3).to_string())

In [ ]:
# ── 2.2 Build G_pmi: clip negatives to 0, keep diagonal = 1 ─────────────────
# Negative nPMI means anti-correlated — we treat as "no edge" (weight=0).
# This is the standard approach: only positive associations inform the graph.

A_pmi = np.where(npmi_mat > 0, npmi_mat, 0.0)
np.fill_diagonal(A_pmi, 1.0)

print("G_pmi adjacency (nPMI clipped to [0,1], diagonal=1):")
print(pd.DataFrame(A_pmi, index=CORE_COLS, columns=CORE_COLS).round(3).to_string())

## 3. G_clin — Curated clinical graph

In [ ]:
# ── 3.1 Clinical graph — manually curated edge weights ───────────────────────
# Weight semantics: 1.0 = strong clinical association (well established)
#                   0.5 = plausible but moderate
#                   0.0 = no known direct association
#
# Rationale for each edge (to be validated / refined by endoscopist):
#
# ULCER ↔ EROSION: 1.0 — erosion is the precursor lesion of ulcer;
#                                    co-existence is gastroenterologically expected.
# ENANTHEMA ↔ ULCER : 1.0 — enanthema (mucosal hyperaemia) is frequently
#                                    present around ulcer margins.
# ENANTHEMA ↔ EROSION : 0.5 — often co-present in gastritis, but enanthema
#                                    can exist without erosion.
# ENANTHEMA ↔ POLYP : 0.5 — hyperaemic mucosa sometimes accompanies polyps,
#                                    especially in inflammatory contexts.
# ULCER ↔ POLYP: 0.5 — ulcer can erode into an inflammatory polyp;
#                                    less common but documented.
# POLYP ↔ EROSION: 0.0 — no established direct clinical relationship.
# POLYP ↔ MICRONODULARITY: 0.5 — both may co-occur in H. pylori gastritis.
# ENANTHEMA ↔ MICRONODULARITY: 0.5 — same as above (H. pylori context).
# MICRONODULARITY ↔ EROSION: 0.0 — no strong direct association.
# MICRONODULARITY ↔ ULCER: 0.0 — no strong direct association.

CLIN_EDGES = {
    ("ENANTEMA",         "PÓLIPO"):            0.5,
    ("ENANTEMA",         "ÚLCERA"):            1.0,
    ("ENANTEMA",         "EROSÃO"):            0.5,
    ("ENANTEMA",         "MICRONODULARIDADE"): 0.5,
    ("PÓLIPO",           "ÚLCERA"):            0.5,
    ("PÓLIPO",           "EROSÃO"):            0.0,
    ("PÓLIPO",           "MICRONODULARIDADE"): 0.5,
    ("ÚLCERA",           "EROSÃO"):            1.0,
    ("ÚLCERA",           "MICRONODULARIDADE"): 0.0,
    ("EROSÃO",           "MICRONODULARIDADE"): 0.0,
}

label2idx = {c: i for i, c in enumerate(CORE_COLS)}

A_clin = np.zeros((K, K), dtype=float)
np.fill_diagonal(A_clin, 1.0)

for (a, b), w in CLIN_EDGES.items():
    i, j = label2idx[a], label2idx[b]
    A_clin[i, j] = w
    A_clin[j, i] = w   # symmetric

print("G_clin adjacency (clinical weights, diagonal=1):")
print(pd.DataFrame(A_clin, index=CORE_COLS, columns=CORE_COLS).round(1).to_string())

print("\nEdges with weight > 0:")
for (a, b), w in sorted(CLIN_EDGES.items(), key=lambda x: -x[1]):
    if w > 0:
        print(f"  {a:22s} ↔ {b:22s}  w={w}")

## 4. GCN normalization (D^-½ TO D^-½) and sparsification

In [ ]:
# ── 4.1 Symmetric degree normalization ───────────────────────────────────────
# Â = D^(-1/2) A D^(-1/2)
# Used in GCN propagation: H' = σ(Â H W)
# Self-loops already included (diagonal=1) so we skip the renorm trick "+I".

def gcn_normalise(A):
    D = np.diag(A.sum(axis=1))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D) + 1e-8))
    return D_inv_sqrt @ A @ D_inv_sqrt

A_emp_norm  = gcn_normalise(A_emp)
A_pmi_norm  = gcn_normalise(A_pmi)
A_clin_norm = gcn_normalise(A_clin)

print("GCN-normalised G_emp:")
print(pd.DataFrame(A_emp_norm, index=CORE_COLS, columns=CORE_COLS).round(3).to_string())

In [ ]:
# ── 4.2 Sparsification: threshold τ ───────────────────── ─────────────────────
# We test τ ∈ {0, 0.1, 0.3} on the raw (pre-normalization) matrices.
# τ=0 → dense graph; τ=0.3 → only strong associations survive.
# For the NB04 ablation we export all 3 variants for all 3 τ values.

def sparsify(A, tau, fill_diagonal=True):
    A_sparse = np.where(A >= tau, A, 0.0)
    if fill_diagonal:
        np.fill_diagonal(A_sparse, 1.0)
    return A_sparse

TAUS = [0.0, 0.1, 0.3]

graphs = {}   # key: (variant, tau)
for tau in TAUS:
    graphs[("emp",  tau)] = gcn_normalise(sparsify(A_emp,  tau))
    graphs[("pmi",  tau)] = gcn_normalise(sparsify(A_pmi,  tau))
    graphs[("clin", tau)] = gcn_normalise(sparsify(A_clin, tau))

# Report edge counts after sparsification
print(f"{'Variant':8s} {'τ':5s}  edges (off-diagonal > 0)")
for (variant, tau), A in graphs.items():
    A_raw = sparsify(
        A_emp if variant=="emp" else A_pmi if variant=="pmi" else A_clin,
        tau)
    off_diag = A_raw - np.diag(np.diag(A_raw))
    n_edges = int((off_diag > 0).sum()) // 2  # undirected
    print(f"  G_{variant:4s}   τ={tau:.1f}  {n_edges} edges")

## 5. Graph visualizations

In [ ]:
# ── 5.1 Layout and helper ────────────────────────── ──────────────────────────
# Node positions on a unit circle, shifted right so MICRONODULARIDADE
# (leftmost node, longest label) has enough horizontal room.
NODE_POS = {
    "ENANTEMA":          ( 0.15,  1.0),
    "PÓLIPO":            ( 1.10,  0.31),
    "ÚLCERA":            ( 0.74, -0.81),
    "EROSÃO":            (-0.44, -0.81),
    "MICRONODULARIDADE": (-0.80,  0.31),
}

NODE_POS_EN = {LABEL_EN[k]: v for k, v in NODE_POS.items()}

NODE_COLOR = "#3498db"
EDGE_CMAP  = plt.cm.YlOrRd

def draw_label_graph(A_raw, labels, pos, title, fname,
                     edge_label_fmt=".2f", node_size=2200,
                     min_edge_w=0.01):
    G = nx.Graph()
    G.add_nodes_from(labels)

    edges, weights = [], []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            w = A_raw[i, j]
            if w > min_edge_w:
                G.add_edge(labels[i], labels[j], weight=w)
                edges.append((labels[i], labels[j]))
                weights.append(w)

    # Wide canvas + explicit axis limits guarantee labels are never clipped
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.set_facecolor("#f8f9fa")
    fig.patch.set_facecolor("#f8f9fa")

    nx.draw_networkx_nodes(G, pos, node_color=NODE_COLOR,
                           node_size=node_size, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold", ax=ax)

    if weights:
        norm   = mcolors.Normalize(vmin=0, vmax=max(weights))
        colors = [EDGE_CMAP(norm(w)) for w in weights]
        widths = [1.5 + 4.0 * w / max(weights) for w in weights]
        nx.draw_networkx_edges(G, pos, edgelist=edges,
                               edge_color=colors, width=widths,
                               alpha=0.85, ax=ax)
        edge_labels = {(u, v): f"{d['weight']:{edge_label_fmt}}"
                       for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                     font_size=8, ax=ax)

    sm = plt.cm.ScalarMappable(cmap=EDGE_CMAP,
                                norm=mcolors.Normalize(
                                    vmin=0, vmax=max(weights) if weights else 1))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.04, label="Edge weight")

    # Generous margins: left for MICRONODULARITY, right for PÓLIPO
    ax.set_xlim(-2.0, 1.8)
    ax.set_ylim(-1.15, 1.3)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIGS_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

print("Graph drawing helper defined.")

In [ ]:
# ── 5.2 G_emp graph (PT + EN) ──────────────────────── ────────────────────────
# G_emp is asymmetric (P(j|i) ≠ P(i|j)). For visualization we symmetrise via
# (A + A^T)/2 so that each undirected edge shows the mean conditional weight.
# The asymmetric A_emp is preserved unchanged for GCN use in NB04.
A_emp_sym = (A_emp + A_emp.T) / 2.0

draw_label_graph(A_emp_sym, CORE_COLS, NODE_POS,
                 "G_emp — Co-ocorrência Empírica\nMédia P(j|i) + P(i|j) / 2  [simétrico para visualização]",
                 "13_graph_emp_PT.png")

draw_label_graph(A_emp_sym, [LABEL_EN[c] for c in CORE_COLS], NODE_POS_EN,
                 "G_emp — Empirical Co-occurrence\nMean P(j|i) + P(i|j) / 2  [symmetrised for display]",
                 "13_graph_emp_EN.png")

In [ ]:
# ── 5.3 G_pmi graph (PT + EN) ──────────────────────── ────────────────────────
draw_label_graph(A_pmi, CORE_COLS, NODE_POS,
                 "G_pmi — Associação nPMI (positivo)\nAssociação além do acaso",
                 "14_graph_pmi_PT.png")

draw_label_graph(A_pmi, [LABEL_EN[c] for c in CORE_COLS], NODE_POS_EN,
                 "G_pmi — nPMI Association (positive)\nAbove-chance label co-occurrence",
                 "14_graph_pmi_EN.png")

In [ ]:
# ── 5.4 G_clin graph (PT + EN) ─────────────────────── ────────────────────────
draw_label_graph(A_clin, CORE_COLS, NODE_POS,
                 "G_clin — Grafo Clínico Curado\nPesos: 1.0=forte · 0.5=moderado",
                 "15_graph_clin_PT.png", edge_label_fmt=".1f")

draw_label_graph(A_clin, [LABEL_EN[c] for c in CORE_COLS], NODE_POS_EN,
                 "G_clin — Clinically Curated Graph\nWeights: 1.0=strong · 0.5=moderate",
                 "15_graph_clin_EN.png", edge_label_fmt=".1f")

In [ ]:
# ── 5.5 Comparison heatmap: raw adjacency of the 3 graphs (PT) ───────────────
# G_emp shown symmetrised (A+A^T)/2 for visual consistency with G_pmi/G_clin.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
variants = [("G_emp", A_emp_sym), ("G_pmi", A_pmi), ("G_clin", A_clin)]

for ax, (name, A) in zip(axes, variants):
    mask = np.eye(K, dtype=bool)
    sns.heatmap(pd.DataFrame(A, index=CORE_COLS, columns=CORE_COLS),
                mask=mask, annot=True, fmt=".2f", cmap="YlOrRd",
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                cbar_kws={"shrink": 0.7})
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle("Comparação das Matrizes de Adjacência — 3 variantes do grafo",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGS_DIR / "16_adjacency_comparison_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 5.6 Comparison heatmap (EN) ─────────────────────── ────────────────────────
core_en = [LABEL_EN[c] for c in CORE_COLS]
variants_en = [("G_emp", A_emp_sym), ("G_pmi", A_pmi), ("G_clin", A_clin)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, A) in zip(axes, variants_en):
    mask = np.eye(K, dtype=bool)
    sns.heatmap(pd.DataFrame(A, index=core_en, columns=core_en),
                mask=mask, annot=True, fmt=".2f", cmap="YlOrRd",
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                cbar_kws={"shrink": 0.7})
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle("Adjacency Matrix Comparison — 3 graph variants",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGS_DIR / "16_adjacency_comparison_EN.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 5.7 Edge weight comparison bar chart (PT) ────────────────────────────────
# G_emp shown symmetrised for visual consistency with the other two graphs.
pair_labels, w_emp, w_pmi, w_clin = [], [], [], []

for i, a in enumerate(CORE_COLS):
    for j, b in enumerate(CORE_COLS):
        if j <= i:
            continue
        pair_labels.append(f"{a[:4]}\n+{b[:4]}")
        w_emp.append(A_emp_sym[i, j])
        w_pmi.append(A_pmi[i, j])
        w_clin.append(A_clin[i, j])

x = np.arange(len(pair_labels))
w = 0.27

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w, w_emp,  w, label="G_emp",  color="#3498db", edgecolor="black", lw=0.5)
ax.bar(x,     w_pmi,  w, label="G_pmi",  color="#e74c3c", edgecolor="black", lw=0.5)
ax.bar(x + w, w_clin, w, label="G_clin", color="#2ecc71", edgecolor="black", lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(pair_labels, fontsize=8)
ax.set_ylabel("Peso da aresta", fontsize=11)
ax.set_title("Comparação de Pesos por Par de Rótulos — 3 Variantes do Grafo",
             fontsize=13, fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(FIGS_DIR / "17_edge_weights_comparison_PT.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 5.8 Edge weight comparison bar chart (EN) ────────────────────────────────
pair_labels_en = []
for i, a in enumerate(CORE_COLS):
    for j, b in enumerate(CORE_COLS):
        if j <= i:
            continue
        pair_labels_en.append(f"{LABEL_EN[a][:4]}\n+{LABEL_EN[b][:4]}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w, w_emp,  w, label="G_emp",  color="#3498db", edgecolor="black", lw=0.5)
ax.bar(x,     w_pmi,  w, label="G_pmi",  color="#e74c3c", edgecolor="black", lw=0.5)
ax.bar(x + w, w_clin, w, label="G_clin", color="#2ecc71", edgecolor="black", lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(pair_labels_en, fontsize=8)
ax.set_ylabel("Edge weight", fontsize=11)
ax.set_title("Edge Weight Comparison per Label Pair — 3 Graph Variants",
             fontsize=13, fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(FIGS_DIR / "17_edge_weights_comparison_EN.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Export of adjacency matrices

In [ ]:
# ── 6.1 Build and export graphs for ALL seeds ────────────────────────────────
# Each seed gets its own graph directory: graphs/seed{seed}/<variant>_tau{tau}/
# This ensures NB04 loads the correct graph for each seed's training set.

def build_graphs_for_split(df_tr):
    """Compute A_emp, A_pmi, A_clin from a given training DataFrame."""
    N = len(df_tr)
    data_np = df_tr[CORE_COLS].values.astype(float)
    marginals_ = np.array([df_tr[c].sum() for c in CORE_COLS], dtype=float)

    # G_emp
    coocc_ = np.zeros((K, K), dtype=float)
    for i in range(K):
        for j in range(K):
            coocc_[i, j] = float(((data_np[:, i] == 1) & (data_np[:, j] == 1)).sum())
    A_emp_ = np.eye(K, dtype=float)
    for i in range(K):
        for j in range(K):
            if i != j:
                A_emp_[i, j] = coocc_[i, j] / marginals_[i] if marginals_[i] > 0 else 0.0

    # G_pmi
    npmi_ = np.zeros((K, K), dtype=float)
    for i in range(K):
        for j in range(K):
            if i == j:
                npmi_[i, j] = 1.0
                continue
            p_i  = (marginals_[i] + ALPHA) / (N + 2 * ALPHA)
            p_j  = (marginals_[j] + ALPHA) / (N + 2 * ALPHA)
            p_ij = (coocc_[i, j]  + ALPHA) / (N + 4 * ALPHA)
            pmi  = np.log2(p_ij / (p_i * p_j))
            npmi_[i, j] = round(pmi / (-np.log2(p_ij)) if p_ij < 1.0 else 0.0, 4)
    A_pmi_ = np.where(npmi_ > 0, npmi_, 0.0)
    np.fill_diagonal(A_pmi_, 1.0)

    return A_emp_, A_pmi_, A_clin   # A_clin is seed-independent

export_log = {}

for i in range(FOLDS):
    fold_graphs_dir = GRAPHS_DIR / f"fold{i}"
    fold_graphs_dir.mkdir(parents=True, exist_ok=True)

    A_emp_s, A_pmi_s, A_clin_s = build_graphs_for_split(train_splits[i])
    variant_mats_s = {"emp": A_emp_s, "pmi": A_pmi_s, "clin": A_clin_s}

    for variant, A_raw in variant_mats_s.items():
        for tau in TAUS:
            tau_str = str(tau).replace(".", "")
            tag     = f"{variant}_tau{tau_str}"
            out_sub = fold_graphs_dir / tag
            out_sub.mkdir(parents=True, exist_ok=True)

            A_sp  = sparsify(A_raw, tau, fill_diagonal=True)
            A_nrm = gcn_normalise(A_sp)

            np.save(out_sub / "adjacency.npy",      A_sp.astype(np.float32))
            np.save(out_sub / "adjacency_norm.npy", A_nrm.astype(np.float32))
            pd.DataFrame(A_sp,  index=CORE_COLS, columns=CORE_COLS).to_csv(out_sub / "adjacency.csv")
            pd.DataFrame(A_nrm, index=CORE_COLS, columns=CORE_COLS).to_csv(out_sub / "adjacency_norm.csv")

            off    = A_sp - np.diag(np.diag(A_sp))
            n_edges = int((off > 0).sum()) // 2
            full_tag = f"fold{i}/{tag}"
            export_log[full_tag] = {
                "fold": i, "variant": variant, "tau": tau,
                "n_edges": n_edges,
                "density": round(n_edges / (K * (K - 1) / 2), 4),
                "path": str(out_sub),
            }

    print(f"  Fold {i}: exported 9 matrix sets → {fold_graphs_dir}")

# Also keep backward-compatible fold0 at top level (for quick reference)
VARIANT_MATS = {"emp": A_emp, "pmi": A_pmi, "clin": A_clin}
for variant, A_raw in VARIANT_MATS.items():
    for tau in TAUS:
        tau_str = str(tau).replace(".", "")
        out_sub = GRAPHS_DIR / f"{variant}_tau{tau_str}"
        out_sub.mkdir(parents=True, exist_ok=True)
        A_sp  = sparsify(A_raw, tau, fill_diagonal=True)
        A_nrm = gcn_normalise(A_sp)
        np.save(out_sub / "adjacency.npy",      A_sp.astype(np.float32))
        np.save(out_sub / "adjacency_norm.npy", A_nrm.astype(np.float32))
        pd.DataFrame(A_sp,  index=CORE_COLS, columns=CORE_COLS).to_csv(out_sub / "adjacency.csv")
        pd.DataFrame(A_nrm, index=CORE_COLS, columns=CORE_COLS).to_csv(out_sub / "adjacency_norm.csv")

print(f"\nExported per-seed graphs to: {GRAPHS_DIR}/seed{{0,1,2,3,4}}/")
print(f"Backward-compatible fold0 matrices kept at: {GRAPHS_DIR}/<variant>_tau*/")
print(f"Total export entries: {len(export_log)}")

In [ ]:
# ── 6.2 Save label order file (critical for NB04 to load matrices correctly) ─
label_order = {"labels_pt": CORE_COLS, "labels_en": [LABEL_EN[c] for c in CORE_COLS]}
with open(GRAPHS_DIR / "label_order.json", "w", encoding="utf-8") as f:
    json.dump(label_order, f, ensure_ascii=False, indent=2)

print("Saved label_order.json")
print(f"  PT: {CORE_COLS}")
print(f"  EN: {label_order['labels_en']}")

## 7. Consolidation into JSON

In [ ]:
# ── 7.1 Build full JSON report ──────────────────────── ────────────────────────

def mat_to_edge_list(A, labels, min_w=0.001):
    """Return list of {i, j, label_i, label_j, weight} for off-diagonal entries."""
    edges = []
    K = len(labels)
    for i in range(K):
        for j in range(i + 1, K):
            w = float(A[i, j])
            if w > min_w:
                edges.append({
                    "i": i, "j": j,
                    "label_i": labels[i], "label_j": labels[j],
                    "weight": round(w, 6),
                })
    return sorted(edges, key=lambda e: -e["weight"])

# ── Dataset hash (fold_0_train.csv) for reproducibility ─────────────────────
split_path = SPLITS_DIR / "fold_0_train.csv"
with open(split_path, "rb") as f:
    split_hash = hashlib.sha256(f.read()).hexdigest()

# ── Build report ────────────────────────────── ───────────────────────────────
report = {
    "notebook": "04_graph_construction",
    "description": "Label co-occurrence graphs for relational multilabel modelling ()",
    "source_split": "fold_0_train.csv",
    "source_split_sha256": split_hash,
    "n_train_images": N_train,
    "labels_pt": CORE_COLS,
    "labels_en": [LABEL_EN[c] for c in CORE_COLS],
    "K": K,
    "alpha_laplace": ALPHA,
    "taus_tested": TAUS,

    # Per-label statistics
    "label_prevalence": {
        c: {
            "count": int(df_train[c].sum()),
            "prevalence": round(float(df_train[c].mean()), 4),
        }
        for c in CORE_COLS
    },

    # Raw co-occurrence matrix (upper triangle)
    "coocc_raw_upper": [
        {"i": i, "j": j,
         "label_i": CORE_COLS[i], "label_j": CORE_COLS[j],
         "n_ij": int(coocc_raw[i, j])}
        for i in range(K) for j in range(i+1, K)
    ],

    # nPMI values ​​(upper triangle)
    "npmi_upper": [
        {"i": i, "j": j,
         "label_i": CORE_COLS[i], "label_j": CORE_COLS[j],
         "pmi": float(pmi_raw[i, j]),
         "npmi": float(npmi_mat[i, j])}
        for i in range(K) for j in range(i+1, K)
    ],

    # Clinical edge weights
    "clin_edges": [
        {"label_i": a, "label_j": b, "weight": w, "rationale": "see cell 3.1"}
        for (a, b), w in sorted(CLIN_EDGES.items(), key=lambda x: -x[1])
    ],

    # Per-variant, per-τ edge lists (raw, pre-normalization)
    "graphs": {},

    # Export log
    "export_log": export_log,
}

for variant, A_raw in VARIANT_MATS.items():
    report["graphs"][variant] = {}
    for tau in TAUS:
        tau_str = str(tau).replace(".", "")
        A_sparse = sparsify(A_raw, tau, fill_diagonal=True)
        off = A_sparse - np.diag(np.diag(A_sparse))
        n_edges = int((off > 0).sum()) // 2
        density = round(n_edges / (K * (K - 1) / 2), 4)
        report["graphs"][variant][f"tau{tau_str}"] = {
            "tau": tau,
            "n_edges": n_edges,
            "density": density,
            "edges": mat_to_edge_list(A_sparse, CORE_COLS),
        }

# ── Save ─────────────────────────────────── ───────────────────────────────────
out_path = RESULTS_DIR / "04_graph_stats.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Saved: {out_path}")
print(f"  Variants: {list(VARIANT_MATS.keys())}")
print(f"  τ values: {TAUS}")
print(f"  Source split hash (sha256): {split_hash[:16]}...")

## 8. Executive summary

In [ ]:
# ── 8. Summary ──────────────────────────────── ────────────────────────────────
print("=" * 70)
print("NB04 — GRAPH CONSTRUCTION — SUMMARY")
print("=" * 70)

print(f"\nTraining set (seed 42): {N_train} images")
print(f"Labels (K={K}): {', '.join(CORE_COLS)}")

print("\nPrevalence:")
for c in CORE_COLS:
    n = int(df_train[c].sum())
    print(f"  {c:25s}: {n:4d} ({n/N_train*100:.1f}%)")

print("\nGraph variants (τ=0, dense):")
for variant, A_raw in VARIANT_MATS.items():
    off = A_raw - np.diag(np.diag(A_raw))
    n_pos = int((off > 0).sum()) // 2
    n_zero = int((off == 0).sum()) // 2
    print(f"  G_{variant:4s}: {n_pos} positive edges, {n_zero} zero edges")

print("\nSparsification effect (G_emp):")
for tau in TAUS:
    A_sp = sparsify(A_emp, tau)
    off  = A_sp - np.diag(np.diag(A_sp))
    n    = int((off > 0).sum()) // 2
    print(f"  τ={tau:.1f}: {n} edges ({n/(K*(K-1)//2)*100:.0f}% of max {K*(K-1)//2})")

print("\nKey clinical associations (G_clin):")
for (a, b), w in sorted(CLIN_EDGES.items(), key=lambda x: -x[1]):
    if w > 0:
        strength = "strong" if w == 1.0 else "moderate"
        print(f"  {a:22s} ↔ {b:22s}  w={w}  ({strength})")

print("\nOutputs:")
print(f"  Figures    : {FIGS_DIR}")
print(f"    13_graph_emp_PT/EN.png")
print(f"    14_graph_pmi_PT/EN.png")
print(f"    15_graph_clin_PT/EN.png")
print(f"    16_adjacency_comparison_PT/EN.png")
print(f"    17_edge_weights_comparison_PT/EN.png")
print(f"  Matrices   : {GRAPHS_DIR}")
for (variant, tau), _ in sorted(graphs.items()):
    tau_str = str(tau).replace(".", "")
    print(f"    {variant}_tau{tau_str}/adjacency{{,_norm}}.{{npy,csv}}")
print(f"  label_order: {GRAPHS_DIR}/label_order.json")
print(f"  JSON report: {RESULTS_DIR}/04_graph_stats.json")

print("\nNext: NB04 — Relational models (M1 chains, M2 reg, M3 ML-GCN)")
print("      Load adjacency_norm.npy from graphs/<variant>_tau<τ>/")
print("=" * 70)